## Завдання 1. Setup і завантаження даних

In [1]:
import sys
# pgserver має wheel-и лише для Python 3.9–3.12
assert sys.version_info < (3, 13), (
    f'Python {sys.version.split()[0]}: pgserver потребує Python 3.9–3.12. '
    'Colab: Runtime → Change runtime type → старіша Runtime version; локально: python3.12 -m venv .venv'
)

%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

import os
import io
import csv
import re
import time
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
# Явно psycopg2: нові версії SQLAlchemy для 'postgresql://' обирають psycopg3
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 112.5 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

def q(sql: str) -> pd.DataFrame:
    """SELECT -> DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql: str) -> None:
    """DDL / DML (кілька statement-ів) в одній транзакції напряму через psycopg2-cursor.
    Без параметрів, тому '%' (modulo в seed-SQL) не трактується як placeholder."""
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql)
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()

PLANS = {}   # усі EXPLAIN-output-и зберігаються тут для зведеної таблиці

def explain(sql: str, label: str = None) -> str:
    """Виконати EXPLAIN (ANALYZE, BUFFERS) і надрукувати план."""
    with engine.connect() as conn:
        plan = '\n'.join(row[0] for row in conn.execute(text(sql)))
    print(plan)
    if label:
        PLANS[label] = plan
    return plan

### Варіант A. NYC Taxi з офіційного Parquet

Відхилення від шаблону, і чому:
1. **Два місяці (лютий + березень 2024), а не `head(500_000)` одного місяця.** Березень 2024 має ≈ 3,5 млн поїздок, тобто 500 000 перших рядків — це лише перші 4–5 днів.
   На такій історії `LAG(trip_count, 30)` завжди `NULL`, і фінальний view (`WHERE trips_lag_30d IS NOT NULL`) був би порожнім. 29 + 31 = 60 днів — як у stub-схемі.
2. **Фільтр періоду.** TLC-файли містять поодинокі поїздки з «чужими» датами (2002, 2009, кінець попереднього місяця) — вони дали б зони з одним днем у 2009 році.
3. **Швидке завантаження.** Після очищення ≈ 5–6 млн рядків; `to_sql` з `method=psql_insert_copy` пише їх через `COPY FROM STDIN`, а не тисячами `INSERT` (стандартний рецепт з документації pandas).

URL у шаблоні обгорнутий у `< >` — це артефакт форматування, `urlretrieve` з таким рядком не працює.

`DROP TABLE ... t7_hw_taxi_daily` має `CASCADE`: `pgserver` з `cleanup_mode='stop'` зберігає базу між перезапусками kernel, і без `CASCADE` повторний Run All падає на залежному view `v_t7_hw_taxi_features`.

In [3]:
FORCE_STUB = os.environ.get('HW7_FORCE_STUB') == '1'

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

MONTHS = ['2024-02', '2024-03']
URL_TEMPLATE = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet'
PERIOD_START, PERIOD_END = pd.Timestamp('2024-02-01'), pd.Timestamp('2024-04-01')   # [start, end)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

DATA_MODE = 'stub'
if not FORCE_STUB:
    try:
        frames = []
        for month in MONTHS:
            parquet_path = DATA_DIR / f'yellow_tripdata_{month}.parquet'
            if not parquet_path.exists():                    # кеш для повторного Run All
                urlretrieve(URL_TEMPLATE.format(month=month), parquet_path)
            frames.append(pd.read_parquet(parquet_path, columns=columns))
        df = pd.concat(frames, ignore_index=True)
        DATA_MODE = 'taxi'
    except Exception as exc:
        print('TLC недоступний, переходимо на stub-схему:', repr(exc)[:200])

print('DATA_MODE =', DATA_MODE)

DATA_MODE = taxi


In [4]:
def psql_insert_copy(table, conn, keys, data_iter):
    """pandas.to_sql(method=...) через COPY FROM STDIN (рецепт з документації pandas)."""
    dbapi_conn = conn.connection
    with dbapi_conn.cursor() as cur:
        buf = io.StringIO()
        csv.writer(buf).writerows(data_iter)
        buf.seek(0)
        columns_sql = ', '.join(f'"{k}"' for k in keys)
        table_sql = f'{table.schema}.{table.name}' if table.schema else table.name
        cur.copy_expert(f'COPY {table_sql} ({columns_sql}) FROM STDIN WITH CSV', buf)

if DATA_MODE == 'taxi':
    n_before = len(df)

    # Basic cleaning before loading to PostgreSQL (як у шаблоні) + фільтр періоду
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    df = df[df['PULocationID'].notna()]
    df = df[(df['tpep_pickup_datetime'] >= PERIOD_START) & (df['tpep_pickup_datetime'] < PERIOD_END)]
    print(f'rows: {n_before:,} -> {len(df):,} after cleaning')

    t0 = time.perf_counter()
    df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False,
              chunksize=200_000, method=psql_insert_copy)
    print(f'loaded in {time.perf_counter() - t0:.1f} s')
    del df, frames

    display(q('SELECT COUNT(*) AS n_rows, MIN(tpep_pickup_datetime) AS min_ts, MAX(tpep_pickup_datetime) AS max_ts FROM nyc_taxi_raw'))

rows: 6,590,154 -> 6,341,305 after cleaning
loaded in 45.1 s


,n_rows,min_ts,max_ts
0,6341305,2024-02-01,2024-03-31 23:59:59


### Daily aggregation `t7_hw_taxi_daily` (режим `taxi`)
SQL — як у шаблоні. Після нього — **densification**: у реальних TLC-даних малі зони мають дні без жодної поїздки, і такого дня просто немає в агрегаті.
`LAG(trip_count, 7)` рахує **рядки**, а не дні, тож для зони з пропусками «lag-7d» насправді брав би значення 8, 9 чи 15 днів тому — тиха ML-помилка, схожа на відсутній `PARTITION BY`.
Тому додаємо відсутні пари (zone, day) з `trip_count = 0`, `total_distance = 0` і `avg_fare = NULL` (у день без поїздок середнього тарифу немає; `AVG`/`STDDEV_SAMP` у вікнах ігнорують `NULL`).
Після цього кожна зона має рівно 60 рядків, і «N рядків назад» = «N днів назад».

In [5]:
TAXI_DAILY_SQL = '''
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;  -- CASCADE: view v_t7_hw_taxi_features з попереднього запуску

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
'''

DENSIFY_SQL = '''
INSERT INTO t7_hw_taxi_daily (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
SELECT z.pickup_zone_id, d.ts_day::DATE, 0, NULL, 0
FROM (SELECT DISTINCT pickup_zone_id FROM t7_hw_taxi_daily) AS z
CROSS JOIN generate_series(
    (SELECT MIN(ts_day) FROM t7_hw_taxi_daily),
    (SELECT MAX(ts_day) FROM t7_hw_taxi_daily),
    INTERVAL '1 day'
) AS d(ts_day)
ON CONFLICT (pickup_zone_id, ts_day) DO NOTHING;
'''

if DATA_MODE == 'taxi':
    run(TAXI_DAILY_SQL)
    gaps_before = q('''
        SELECT COUNT(*) AS n_rows,
               COUNT(DISTINCT pickup_zone_id) AS n_zones,
               COUNT(DISTINCT ts_day) AS n_days,
               COUNT(DISTINCT pickup_zone_id) * COUNT(DISTINCT ts_day) - COUNT(*) AS missing_zone_days
        FROM t7_hw_taxi_daily
    ''')
    display(gaps_before.assign(stage='before densify'))
    run(DENSIFY_SQL)
    display(q('''
        SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pickup_zone_id) AS n_zones, COUNT(DISTINCT ts_day) AS n_days,
               COUNT(*) FILTER (WHERE trip_count = 0) AS zero_fill_rows, 'after densify' AS stage
        FROM t7_hw_taxi_daily
    '''))

,n_rows,n_zones,n_days,missing_zone_days,stage
0,13511,258,60,1969,before densify


,n_rows,n_zones,n_days,zero_fill_rows,stage
0,15480,258,60,1969,after densify


Production-style демонстрація `TIMESTAMPTZ` (тільки в режимі `taxi`):

In [6]:
if DATA_MODE == 'taxi':
    run('''
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2) AS trip_distance,
        fare_amount::NUMERIC(10, 2) AS fare_amount,
        passenger_count::INTEGER AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100;
    ''')
    display(q('''
    SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5
    '''))

,pickup_at,pickup_type
0,2024-02-01 05:04:45+00:00,timestamp with time zone
1,2024-02-01 05:56:31+00:00,timestamp with time zone
2,2024-02-01 05:07:50+00:00,timestamp with time zone
3,2024-02-01 05:01:49+00:00,timestamp with time zone
4,2024-02-01 05:37:35+00:00,timestamp with time zone


### Варіант B. Stub-schema fallback (режим `stub`)

In [7]:
STUB_SQL = '''
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;  -- CASCADE: view v_t7_hw_taxi_features з попереднього запуску

CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id  INTEGER NOT NULL,
    ts_day          DATE    NOT NULL,
    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);

INSERT INTO t7_hw_taxi_daily (
    pickup_zone_id,
    ts_day,
    trip_count,
    avg_fare,
    total_distance
)
SELECT
    z.zone_id,
    DATE '2024-03-31' - (59 - n) AS ts_day,
    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
FROM generate_series(0, 59) AS n
CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);
'''

if DATA_MODE == 'stub':
    run(STUB_SQL)
    print('stub-схема створена: 60 днів × 5 zones')

Перевірка (обидва режими): скільки днів має кожна зона.

In [8]:
q('''
SELECT pickup_zone_id, COUNT(*) AS n_days, MIN(ts_day) AS first_day, MAX(ts_day) AS last_day
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;
''')

,pickup_zone_id,n_days,first_day,last_day
0,1,60,2024-02-01,2024-03-31
1,2,60,2024-02-01,2024-03-31
2,3,60,2024-02-01,2024-03-31
3,4,60,2024-02-01,2024-03-31
4,6,60,2024-02-01,2024-03-31
5,7,60,2024-02-01,2024-03-31
6,8,60,2024-02-01,2024-03-31
7,9,60,2024-02-01,2024-03-31
8,10,60,2024-02-01,2024-03-31
9,11,60,2024-02-01,2024-03-31


In [9]:
q('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    COUNT(DISTINCT ts_day) AS n_days,
    MIN(n_days_per_zone) AS min_days_per_zone,
    MAX(n_days_per_zone) AS max_days_per_zone
FROM (
    SELECT pickup_zone_id, ts_day, COUNT(*) OVER (PARTITION BY pickup_zone_id) AS n_days_per_zone
    FROM t7_hw_taxi_daily
) AS t;
''')

,n_rows,n_zones,n_days,min_days_per_zone,max_days_per_zone
0,15480,258,60,60,60


**Висновок.** `t7_hw_taxi_daily` має значно більше за 100 рядків (stub: 300; taxi: ≈ 260 зон × 60 днів ≈ 15 тис.), і `min_days_per_zone = max_days_per_zone = 60`:
календар щільний, тому row-based `LAG` коректно інтерпретується як «N днів тому».

> ✏️ Звірте числа з виводом (кількість зон і пропущених zone-days у режимі `taxi`).

## Завдання 2. Time-series ML features
Усі вікна мають `PARTITION BY pickup_zone_id`. `ORDER BY ts_day::TIMESTAMP` у `RANGE`-вікнах — щоб `INTERVAL`-offset був однозначним.

### Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [10]:
q('''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-01,2,NaN,NaN,None
1,1,2024-02-02,0,2.0,NaN,None
2,1,2024-02-03,0,0.0,NaN,None
3,1,2024-02-04,3,0.0,NaN,None
4,1,2024-02-05,1,3.0,NaN,None
5,1,2024-02-06,0,1.0,NaN,None
6,1,2024-02-07,1,0.0,NaN,None
7,1,2024-02-08,3,1.0,2.0,None
8,1,2024-02-09,1,3.0,0.0,None
9,1,2024-02-10,0,1.0,0.0,None


Перші 1/7/30 рядків кожної зони мають `NULL` у відповідному lag — історії ще немає. Для training dataset з повною історією такі рядки відфільтровуються (`trips_lag_30d IS NOT NULL`).

### Feature 2. Rolling 7-day average fare

In [11]:
q('''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-01,117.50,117.500000,1
1,1,2024-02-02,NaN,117.500000,2
2,1,2024-02-03,NaN,117.500000,3
3,1,2024-02-04,45.10,81.300000,4
4,1,2024-02-05,125.00,95.866667,5
5,1,2024-02-06,NaN,95.866667,6
6,1,2024-02-07,110.00,99.400000,7
7,1,2024-02-08,92.40,93.125000,7
8,1,2024-02-09,10.00,76.500000,7
9,1,2024-02-10,NaN,76.500000,7


`days_in_window` росте 1 → 7 на початку історії зони й далі стабільно дорівнює 7: `INTERVAL '6 days' PRECEDING` + поточний день = 7 календарних днів.

### Feature 3. Rolling 30-day fare volatility

In [12]:
q('''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-01,117.50,NaN,1
1,1,2024-02-02,NaN,NaN,2
2,1,2024-02-03,NaN,NaN,3
3,1,2024-02-04,45.10,51.194531,4
4,1,2024-02-05,125.00,44.124861,5
5,1,2024-02-06,NaN,44.124861,6
6,1,2024-02-07,110.00,36.714302,7
7,1,2024-02-08,92.40,31.949257,8
8,1,2024-02-09,10.00,45.905018,9
9,1,2024-02-10,NaN,45.905018,10


На першому дні зони `STDDEV_SAMP` = `NULL` (вибіркове стандартне відхилення з одного значення не визначене) — це очікувано, а не помилка.

### Feature 4. Expanding cumulative trips

In [13]:
q('''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-01,2,2,2.000000
1,1,2024-02-02,0,2,1.000000
2,1,2024-02-03,0,2,0.666667
3,1,2024-02-04,3,5,1.250000
4,1,2024-02-05,1,6,1.200000
5,1,2024-02-06,0,6,1.000000
6,1,2024-02-07,1,7,1.000000
7,1,2024-02-08,3,10,1.250000
8,1,2024-02-09,1,11,1.222222
9,1,2024-02-10,0,11,1.100000


### Combined feature view

In [14]:
run('''
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
''')

In [15]:
q('''
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,2,95.00,18.43,1,2,2,108.918333,36.035289,60,1.935484
1,1,2024-03-03,2,51.00,0.09,2,4,0,108.985000,35.911127,62,1.937500
2,1,2024-03-04,1,40.00,0.01,2,1,0,87.318333,36.194015,63,1.909091
3,1,2024-03-05,1,120.00,0.03,1,1,3,87.526667,36.153623,64,1.882353
4,1,2024-03-06,0,NaN,0.00,1,5,1,82.200000,35.922024,64,1.828571
5,1,2024-03-07,0,NaN,0.00,0,0,0,82.200000,35.922024,64,1.777778
6,1,2024-03-08,2,82.55,10.20,0,1,1,77.710000,35.474537,66,1.783784
7,1,2024-03-09,0,NaN,0.00,2,2,3,73.387500,36.166693,66,1.736842
8,1,2024-03-10,3,60.33,3.33,0,2,1,75.720000,33.188259,69,1.769231
9,1,2024-03-11,3,44.33,19.61,3,1,0,76.802500,33.416543,72,1.800000


Перевірка, що всі window definitions у view містять `PARTITION BY pickup_zone_id`, і обсяг training-набору:

In [16]:
viewdef = q("SELECT pg_get_viewdef('v_t7_hw_taxi_features'::regclass, true) AS def")['def'].iloc[0]
windows = re.findall(r'(\w+) AS \(\s*PARTITION BY ([\w.]+)', viewdef)
print('window -> partition key:', windows)
assert len(windows) == 4 and all(col.endswith('pickup_zone_id') for _, col in windows)

q('''
SELECT COUNT(*) AS n_training_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day, MAX(ts_day) AS last_day
FROM v_t7_hw_taxi_features
''')

window -> partition key: [('w_pid', 't7_hw_taxi_daily.pickup_zone_id'), ('w7', 't7_hw_taxi_daily.pickup_zone_id'), ('w30', 't7_hw_taxi_daily.pickup_zone_id'), ('we', 't7_hw_taxi_daily.pickup_zone_id')]


,n_training_rows,n_zones,first_day,last_day
0,7740,258,2024-03-02,2024-03-31


### Чому `PARTITION BY` обов'язковий: демонстрація тихої помилки

In [17]:
q('''
WITH with_and_without AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS lag_1d_correct,
        LAG(trip_count, 1) OVER (ORDER BY pickup_zone_id, ts_day)            AS lag_1d_no_partition,
        ROW_NUMBER()       OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS day_idx
    FROM t7_hw_taxi_daily
)
SELECT pickup_zone_id, ts_day, trip_count, lag_1d_correct, lag_1d_no_partition
FROM with_and_without
WHERE day_idx = 1
ORDER BY pickup_zone_id
LIMIT 5;
''')

,pickup_zone_id,ts_day,trip_count,lag_1d_correct,lag_1d_no_partition
0,1,2024-02-01,2,None,NaN
1,2,2024-02-01,0,None,3.0
2,3,2024-02-01,2,None,0.0
3,4,2024-02-01,70,None,6.0
4,6,2024-02-01,0,None,345.0


**Interpretation.** На першому дні кожної зони правильний lag = `NULL` (історії немає), а без `PARTITION BY` lag підтягує `trip_count` **останнього дня попередньої зони**.
Модель отримала б «історію» іншої сутності, і жодна помилка не виникла б — тому це тиха ML-помилка. Для першої зони обидва варіанти `NULL`, тож проблема видна лише з другої зони.

У view після фільтра `trips_lag_30d IS NOT NULL` кожна зона має 30 рядків (дні 31–60): перші 30 днів пішли на «прогрів» найдовшого lag.
У stub-даних `trip_count` росте лінійно з `n` і коливається з періодом 5, `avg_fare` — з періодом 4, тому `rolling_std_fare_30d` майже константна, а rolling 7d середнє згладжує 4-денний цикл.
На реальних TLC-даних у lag-7d видно тижневу сезонність (вихідні vs будні), а в `rolling_std_fare_30d` — підвищену волатильність у зонах аеропортів.

> ✏️ Підставте спостереження з вашого виводу (конкретні зони/значення).

## Завдання 3. JSONB-таблиця для подій

In [18]:
EVENTS_SQL = '''
DROP TABLE IF EXISTS t7_hw_events CASCADE;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
'''
run(EVENTS_SQL)

In [19]:
q('''
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
''')

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


Структура payload різна для кожного типу — перелік top-level ключів:

In [20]:
q('''
SELECT
    event_type,
    STRING_AGG(DISTINCT k, ', ' ORDER BY k) AS payload_keys
FROM t7_hw_events,
     LATERAL jsonb_object_keys(payload) AS k
GROUP BY event_type
ORDER BY event_type;
''')

,event_type,payload_keys
0,add_to_cart,"price, product_id, quantity, tags"
1,checkout,"items_count, order_id, payment_method, total"
2,click,"device, element_id, page_url"
3,page_view,"device, referrer, url"
4,recommendation_shown,"experiment, model_name, model_version, recommendations"


### JSONB Query 1. Containment `@>`
**Питання:** які checkout-и оплачені карткою?

In [21]:
q('''
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;
''')

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


`@>` перевіряє, що payload **містить** вказану пару ключ–значення (з урахуванням типу: `'card'` — рядок). Тут збіг — третина checkout-ів, бо `payment_method` циклічно бере одне з трьох значень.

### JSONB Query 2. Key existence `?` та `?|`
**Питання:** які події мають теги та які — ML-атрибути рекомендації?

In [22]:
q('''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;
''')

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [23]:
q('''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;
''')

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


In [24]:
q('''
SELECT
    COUNT(*) FILTER (WHERE payload ? 'tags')                                   AS n_with_tags,
    COUNT(*) FILTER (WHERE payload ?| ARRAY['experiment', 'model_version'])    AS n_with_any_ml_key,
    COUNT(*) FILTER (WHERE payload ?& ARRAY['experiment', 'model_version'])    AS n_with_all_ml_keys,
    COUNT(*) FILTER (WHERE payload ? 'device')                                 AS n_with_device
FROM t7_hw_events;
''')

,n_with_tags,n_with_any_ml_key,n_with_all_ml_keys,n_with_device
0,10000,10000,10000,20000


`?` — чи є top-level ключ; `?|` — чи є **хоча б один** з переліку; `?&` — **усі**. `tags` має лише `add_to_cart` (10 000 подій), `experiment`/`model_version` — лише `recommendation_shown`,
тому `?|` і `?&` тут дають однакову кількість. `device` є і в `page_view`, і в `click` — один ключ у різних типах подій, що важливо для Завдання 4.
Оператори працюють лише з top-level ключами; для вкладених полів потрібен `@>` з вкладеним об'єктом або JSONPath.

### JSONB Query 3. JSONPath
**Питання:** які товари рекомендувала модель і які checkout-и мали суму понад 300?

In [25]:
q('''
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;
''')

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [26]:
q('''
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;
''')

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


`jsonb_path_query` — set-returning function: одна подія з трьома рекомендаціями розгортається в три рядки (тому 15 рядків = 5 подій). Це готовий формат для «exploded» таблиці показів рекомендацій.
Фільтр `$.total ? (@ > 300)` порівнює **число** всередині JSON без явного cast-у; `->>` повертає текст, тому для арифметики потрібен `::NUMERIC`.

## Завдання 4. GIN-index demo + EXPLAIN ANALYZE
### Step 1. Containment без GIN-індексу

In [27]:
run('''
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
''')

In [28]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''', label='1. @> payment_method=card, no GIN')

Aggregate  (cost=1754.12..1754.13 rows=1 width=8) (actual time=33.726..33.727 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=1649 width=0) (actual time=0.016..33.510 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.196 ms
Execution Time: 33.752 ms


### Step 2. GIN-індекс (`jsonb_path_ops`) для containment / JSONPath

In [29]:
run('''
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
''')

In [30]:
q('''
SELECT
    pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
    pg_size_pretty(pg_relation_size('t7_hw_events'))                      AS table_size;
''')

,gin_index_size,table_size
0,1144 kB,8000 kB


### Step 3. Той самий containment після GIN

In [31]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''', label='2. @> payment_method=card, GIN path_ops')

Aggregate  (cost=1091.66..1091.67 rows=1 width=8) (actual time=5.322..5.324 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=34.19..1083.45 rows=3284 width=0) (actual time=0.524..5.128 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..33.37 rows=3284 width=0) (actual time=0.362..0.362 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.295 ms
Execution Time: 5.354 ms


In [32]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
''', label='2b. @> referrer=google, GIN path_ops')

Aggregate  (cost=1064.17..1064.18 rows=1 width=8) (actual time=3.800..3.801 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.67..1058.53 rows=2257 width=0) (actual time=0.431..3.650 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.11 rows=2257 width=0) (actual time=0.290..0.290 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.154 ms
Execution Time: 3.826 ms


Додатково — `?` з GIN `jsonb_path_ops`: цей opclass **не підтримує** операторів існування ключа, тому план залишається `Seq Scan`.
Для `?`, `?|`, `?&` потрібен default `jsonb_ops`:

In [33]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM t7_hw_events WHERE payload ? 'tags';
''', label='2c. ? tags, only GIN path_ops')

run('''
CREATE INDEX idx_t7_events_payload_gin_ops ON t7_hw_events USING GIN (payload);  -- jsonb_ops (default)
ANALYZE t7_hw_events;
''')

plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM t7_hw_events WHERE payload ? 'tags';
''', label='2d. ? tags, GIN jsonb_ops')

display(q('''
SELECT indexrelid::regclass AS index_name, pg_size_pretty(pg_relation_size(indexrelid)) AS size
FROM pg_index WHERE indrelid = 't7_hw_events'::regclass ORDER BY 1
'''))

# прибираємо jsonb_ops, щоб далі порівнювати рівно ті індекси, що в умові
run('DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops; ANALYZE t7_hw_events;')

Aggregate  (cost=1650.12..1650.13 rows=1 width=8) (actual time=10.041..10.042 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10050 width=0) (actual time=0.006..9.521 rows=10000 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 40000
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=2
Planning Time: 0.075 ms
Execution Time: 10.058 ms
Aggregate  (cost=1230.23..1230.24 rows=1 width=8) (actual time=4.586..4.587 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=78.35..1204.92 rows=10125 width=0) (actual time=0.752..4.097 rows=10000 loops=1)
        Recheck Cond: (payload ? 'tags'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_ops  (cost=0.00..75.82 rows=10125 width=0) (actual time=0.635..0.636 rows=10000 loops=1)
              Index Cond: (payload ? 'tags'::text)
        

,index_name,size
0,t7_hw_events_pkey,1112 kB
1,idx_t7_events_payload_gin_pathops,1144 kB
2,idx_t7_events_payload_gin_ops,1696 kB


### Step 4. Коли GIN не допомагає: path-specific text equality

In [34]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''', label='3. ->> device = mobile, no expression index')

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=9.233..9.234 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.017..8.794 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=6
Planning Time: 0.136 ms
Execution Time: 9.263 ms


In [35]:
run('''
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
''')

In [36]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''', label='4. ->> device = mobile, expression B-tree')

Aggregate  (cost=1243.13..1243.14 rows=1 width=8) (actual time=3.763..3.764 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.00..1222.25 rows=8350 width=0) (actual time=0.503..3.268 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.92 rows=8350 width=0) (actual time=0.352..0.352 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.217 ms
Execution Time: 3.791 ms


Той самий бізнес-фільтр можна переписати через containment — тоді спрацює вже наявний GIN (але семантика трохи інша: `@>` порівнює JSON-значення з типом, `->>` — текст):

In [37]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"device": "mobile"}';
''', label='4b. @> device=mobile, GIN path_ops')

Aggregate  (cost=1170.07..1170.08 rows=1 width=8) (actual time=5.427..5.428 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=59.70..1151.68 rows=7358 width=0) (actual time=1.791..5.023 rows=8333 loops=1)
        Recheck Cond: (payload @> '{"device": "mobile"}'::jsonb)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..57.86 rows=7358 width=0) (actual time=1.669..1.669 rows=8333 loops=1)
              Index Cond: (payload @> '{"device": "mobile"}'::jsonb)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=8
Planning Time: 0.176 ms
Execution Time: 5.455 ms


### Write-cost: вставка 50 000 подій у таблицю без індексів і з індексами

In [38]:
def timed_insert(index_ddl: str) -> float:
    run('''
    DROP TABLE IF EXISTS t7_hw_events_writetest;
    CREATE TABLE t7_hw_events_writetest (LIKE t7_hw_events INCLUDING DEFAULTS);
    ''' + index_ddl)
    t0 = time.perf_counter()
    run('INSERT INTO t7_hw_events_writetest (event_id, user_id, event_ts, event_type, payload) '
        'SELECT event_id, user_id, event_ts, event_type, payload FROM t7_hw_events;')
    return time.perf_counter() - t0

write_cost = pd.DataFrame([
    ('no indexes',                   timed_insert('')),
    ('GIN jsonb_path_ops',           timed_insert('CREATE INDEX ON t7_hw_events_writetest USING GIN (payload jsonb_path_ops);')),
    ('GIN jsonb_ops',                timed_insert('CREATE INDEX ON t7_hw_events_writetest USING GIN (payload);')),
    ('B-tree (payload->>device)',    timed_insert("CREATE INDEX ON t7_hw_events_writetest ((payload ->> 'device'));")),
], columns=['indexes', 'insert_50k_seconds'])
run('DROP TABLE IF EXISTS t7_hw_events_writetest;')
write_cost.assign(slowdown=lambda d: (d.insert_50k_seconds / d.insert_50k_seconds.iloc[0]).round(2))

,indexes,insert_50k_seconds,slowdown
0,no indexes,0.085182,1.00
1,GIN jsonb_path_ops,0.179363,2.11
2,GIN jsonb_ops,0.517191,6.07
3,B-tree (payload->>device),0.148183,1.74


### Зведення EXPLAIN-планів Завдання 4

In [39]:
def plan_summary(label: str, plan: str) -> dict:
    first = plan.splitlines()[0]
    nodes = re.findall(r'(Seq Scan|Parallel Seq Scan|Bitmap Index Scan|Bitmap Heap Scan|Index Only Scan|Index Scan)', plan)
    cost = re.search(r'cost=[\d.]+\.\.([\d.]+)', first)
    return {
        'scenario': label,
        'scan_nodes': ', '.join(dict.fromkeys(nodes)),
        'total_cost': float(cost.group(1)) if cost else None,
        'execution_ms': float(re.search(r'Execution Time: ([\d.]+) ms', plan).group(1)),
        'shared_hit': sum(int(x) for x in re.findall(r'shared hit=(\d+)', plan.split('Planning')[0])[:1] or [0]),
    }

pd.DataFrame([plan_summary(k, v) for k, v in PLANS.items()])

,scenario,scan_nodes,total_cost,execution_ms,shared_hit
0,"1. @> payment_method=card, no GIN",Seq Scan,1754.13,33.752,1000
1,"2. @> payment_method=card, GIN path_ops","Bitmap Heap Scan, Bitmap Index Scan",1091.67,5.354,1004
2,"2b. @> referrer=google, GIN path_ops","Bitmap Heap Scan, Bitmap Index Scan",1064.18,3.826,1004
3,"2c. ? tags, only GIN path_ops",Seq Scan,1650.13,10.058,1000
4,"2d. ? tags, GIN jsonb_ops","Bitmap Heap Scan, Bitmap Index Scan",1230.24,4.624,1006
5,"3. ->> device = mobile, no expression index",Seq Scan,1750.63,9.263,1000
6,"4. ->> device = mobile, expression B-tree","Bitmap Heap Scan, Bitmap Index Scan",1243.14,3.791,1000
7,"4b. @> device=mobile, GIN path_ops","Bitmap Heap Scan, Bitmap Index Scan",1170.08,5.455,1006


**Interpretation.**
- **Containment до/після GIN (1 → 2):** без індексу — `Seq Scan` по всіх 50 000 рядках, `Rows Removed by Filter: 46666`, `@>` обчислюється для кожного рядка.
  Після `jsonb_path_ops` — `Bitmap Index Scan` по GIN (знаходить 3 334 кандидати) → `Bitmap Heap Scan`; total cost падає, actual time — у кілька разів.
  **Але `Buffers` майже не змінюються (≈ 1 000 сторінок в обох планах, `Heap Blocks: exact=1000`)**: seed розкладає checkout-и рівномірно (`n % 5`), тож збіги є на кожній сторінці таблиці, і heap-скан однаково читає всі сторінки.
  Виграш — у CPU: `@>` перевіряється (recheck) лише для 3 334 рядків замість 50 000. На реальних даних, де події згруповані за часом, виграш у буферах був би більшим.
- **`?` (2c → 2d):** з `jsonb_path_ops` key existence лишається `Seq Scan` — цей opclass хешує *шляхи зі значеннями* і не знає про окремі ключі. Default `jsonb_ops` дає `Bitmap Index Scan`, але більший за розміром.
- **Path equality (3 → 4):** `payload ->> 'device' = 'mobile'` — це рівність над **виразом**, а не над JSONB-документом; GIN не має операторного класу для `->>` + `=`, тому план — `Seq Scan`.
  Після expression B-tree на `(payload ->> 'device')` планувальник використовує `Bitmap Index Scan` по ньому. `device = mobile` має 8 333 рядки (≈ 17 %).
  Важливий побічний ефект: без індексу планувальник **оцінює** `rows=250` (дефолтна селективність 0,5 % для виразу без статистики), а фактично 8 333; після `CREATE INDEX` + `ANALYZE` PostgreSQL збирає статистику по виразу, і оцінка стає ≈ 8 350.
  Тобто expression index покращує ще й кардинальність для подальших JOIN-ів.
- **4b:** той самий фільтр, записаний як `@>`, використовує GIN — але це вже інший запит, і B-tree на конкретному шляху компактніший та підтримує ще й `<`, `>`, `ORDER BY`.

> ✏️ Підставте фактичні total cost, actual time і `Buffers` зі зведеної таблиці у Reflection.

## Завдання 5. JSONB vs normalized
### Step 1. Нормалізована таблиця checkout-подій

In [40]:
run('''
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
''')

In [41]:
q('''
SELECT
    (SELECT COUNT(*) FROM t7_hw_checkouts_norm) AS n_checkouts,
    pg_size_pretty(pg_total_relation_size('t7_hw_checkouts_norm')) AS norm_total_size,
    pg_size_pretty(pg_total_relation_size('t7_hw_events'))         AS events_total_size;
''')

,n_checkouts,norm_total_size,events_total_size
0,10000,1112 kB,10 MB


### Step 2. Один запит двома способами

In [42]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
''', label='5a. JSONB: avg total by payment_method')

Limit  (cost=1954.56..1954.56 rows=3 width=72) (actual time=19.419..19.421 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1954.56..1968.41 rows=5541 width=72) (actual time=19.417..19.419 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.82..1882.94 rows=5541 width=72) (actual time=19.387..19.407 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.97 rows=9990 width=132) (actual time=0.012..11.947 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=14 read=2
Planning Time: 0.148 ms
Execution Time: 19.473

In [43]:
plan = explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
''', label='5b. normalized: avg total by payment_method')

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=5.552..5.554 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=5.550..5.552 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=5.537..5.540 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.010..1.170 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.141 ms
Execution Time: 5.600 ms


Перевірка, що обидва підходи дають однаковий результат:

In [44]:
q('''
SELECT 'jsonb' AS source, payload ->> 'payment_method' AS payment_method,
       ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
UNION ALL
SELECT 'normalized', payment_method, ROUND(AVG(total), 2), COUNT(*)
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY payment_method, source;
''')

,source,payment_method,avg_total,n
0,jsonb,card,300.50,3334
1,normalized,card,300.50,3334
2,jsonb,crypto,300.48,3333
3,normalized,crypto,300.48,3333
4,jsonb,paypal,300.52,3333
5,normalized,paypal,300.52,3333


In [45]:
pd.DataFrame([plan_summary(k, PLANS[k]) for k in ['5a. JSONB: avg total by payment_method',
                                                   '5b. normalized: avg total by payment_method']])

,scenario,scan_nodes,total_cost,execution_ms,shared_hit
0,5a. JSONB: avg total by payment_method,Seq Scan,1954.56,19.473,1000
1,5b. normalized: avg total by payment_method,Seq Scan,269.07,5.600,94


**Interpretation.** Результати агрегації ідентичні. У JSONB-версії план — `Seq Scan` по всій `t7_hw_events` (50 000 рядків, фільтр `event_type = 'checkout'` відкидає ≈ 40 000),
для кожного рядка виконується розбір `->>` і cast у `NUMERIC`, потім `HashAggregate` і `Sort`. Нормалізована версія сканує лише ≈ 10 000 вузьких рядків уже типізованих колонок —
менше сторінок у `Buffers` і менше CPU на рядок. Індекс `idx_t7_checkouts_norm_pm` для цього запиту не потрібен (агрегація по всій таблиці), він корисний для фільтра за одним способом оплати.

> ✏️ Порівняйте фактичні `execution_ms` і `Buffers` у таблиці вище; не стверджуйте, що normalized швидша, якщо у вашому середовищі різниця в межах шуму.

### Step 3. Bonus: MongoDB-equivalent (pseudocode)

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

Кроки pipeline напряму відповідають SQL: `$match` ≈ `WHERE`, `$group` ≈ `GROUP BY` + агрегати, `$sort` + `$limit` ≈ `ORDER BY ... LIMIT`.

- **PostgreSQL JSONB** зручний, коли події треба поєднувати з нормалізованими таблицями (`JOIN` з users/orders), потрібні транзакції, constraints і один SQL-інтерфейс для аналітиків.
- **MongoDB aggregation pipeline** природний для document-first workload: документ — основна одиниця, схема змінюється часто, горизонтальне шардування з коробки.
- **High-volume raw clickstream** (мільярди подій) часто краще жити в document store або data lake (Parquet + S3, ClickHouse/BigQuery), де дешеве сховище й колонковий скан.
- **Curated ML features** зручніше тримати в typed SQL-таблицях: типи, `CHECK`, стабільна схема для training/serving і передбачувані плани.

## Reflection

**GIN до і після.** Containment `payload @> {"payment_method":"card"}` без індексу — `Seq Scan` усіх 50 000 рядків, `Rows Removed by Filter: 46666`.
Після `jsonb_path_ops` план став `Bitmap Index Scan` → `Bitmap Heap Scan`: total cost і actual time зменшились у рази (✏️ підставте числа зі зведення), а `Buffers` лишились ≈ 1 000,
бо синтетичні checkout-и є на кожній сторінці таблиці; виграш дала відмова від перевірки `@>` для 46 тис. рядків. Те саме для `referrer = google` (2 500 рядків).

**Чому GIN не для `->> 'device' = 'mobile'`.** GIN індексує елементи документа для операторів `@>`, `?`, `@?`; вираз `payload ->> 'device'` — це текст, і для `=` над ним GIN не має opclass, тому план лишився `Seq Scan`.
Expression B-tree на `(payload ->> 'device')` прямо індексує цей вираз: планувальник використав індекс, і він ще й підтримує діапазони та `ORDER BY`. Крім того, `jsonb_path_ops` не підтримує `?` — для key existence потрібен `jsonb_ops`, який більший за розміром.

**Read/write trade-off.** GIN пришвидшує containment-фільтри, але кожен `INSERT` розкладає документ на всі ключі/значення й оновлює posting lists: у тесті вставка 50 000 подій з GIN повільніша, ніж без індексів (✏️ коефіцієнт), а `jsonb_ops` ще й займає більше місця. Для write-heavy event-log це суттєво; `fastupdate` пом'якшує вставки ціною повільніших читань до vacuum.

**JSONB чи normalized у ML-інфраструктурі.** JSONB — для сирого event-log із різнорідними й мінливими payload, нових експериментів і полів, які ще невідомо, чи стануть features. Нормалізована таблиця — для стабільних, часто використовуваних атрибутів (сума, спосіб оплати), де потрібні типи, `CHECK` і швидка агрегація: у Step 2 normalized-версія сканувала менше сторінок і не парсила JSON.

**Новий event-type `refund`.** У JSONB — жодної міграції: просто вставляємо `event_type = 'refund'` з payload `{order_id, amount, reason}`; за потреби — `CHECK` на допустимі типи й expression index на нові поля.
У normalized — нова таблиця `t7_hw_refunds_norm` (або колонки в спільній), DDL-міграція, ETL з `t7_hw_events`, FK на checkout за `order_id`, індекси, оновлення feature-запитів.